# SpaNCy Demo — Normalize CyCIF Data & Verify Results

This notebook demonstrates the complete SpaNCy workflow:
1. Load PRAD-CyCIF data
2. Run SpaNCy normalization
3. Verify output quality (non-negativity, distribution alignment)
4. Visualize batch mixing with UMAP before/after
5. Quantify improvement with KS statistics

## 0. Colab Setup

Run these two cells first if using Google Colab.

In [ ]:
# ── Install dependencies (Colab) ──
# torch is pre-installed on Colab; we just need torch-geometric + friends

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# 1. Core scientific stack
install("anndata")
install("scikit-learn")

# 2. torch-geometric — auto-detect torch+CUDA version for compatible wheels
import torch
TORCH_VERSION = torch.__version__.split("+")[0]
CUDA_TAG = torch.version.cuda

if CUDA_TAG:
    cuda_short = "cu" + CUDA_TAG.replace(".", "")
else:
    cuda_short = "cpu"

PYGS_URL = f"https://data.pyg.org/whl/torch-{TORCH_VERSION}+{cuda_short}.html"
print(f"PyG wheel index: {PYGS_URL}")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch-scatter", "torch-sparse",
    "-f", PYGS_URL,
])
install("torch-geometric")

# 3. UMAP for visualization
install("umap-learn")

print(f"\ntorch {torch.__version__}  CUDA {CUDA_TAG}")
import torch_geometric
print(f"torch-geometric {torch_geometric.__version__}")
print("All dependencies installed.")

In [ ]:
# ── Upload spancy.py to Colab ──
# Option A: Upload via browser dialog
from google.colab import files
uploaded = files.upload()  # select spancy.py

# Option B: From Google Drive (uncomment if preferred)
# from google.colab import drive
# drive.mount("/content/drive")
# !cp "/content/drive/MyDrive/path/to/spancy.py" .

import os
assert os.path.exists("spancy.py"), "spancy.py not found — upload it first"
print(f"spancy.py ready ({os.path.getsize('spancy.py'):,} bytes)")

## 1. Load Data

In [ ]:
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.stats import ks_2samp

from spancy import (
    DEFAULT_CYCLE_CONFIG, load_adata, train, normalize_adata,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)

# Upload your .h5ad or point to a Drive path
DATA_PATH = "PRAD_anndata.h5ad"     # <-- change this
OUTPUT_PATH = "PRAD_normalized.h5ad"

adata = load_adata(DATA_PATH)
print(adata)
print(f"Markers: {list(adata.var_names)}")
print(f"Batches: {adata.obs['batch'].unique().tolist()}")

## 2. Run SpaNCy Normalization

One call trains the model, a second applies it. Adjust `DEVICE` and `N_EPOCHS` as needed.

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_EPOCHS = 100
print(f"Using device: {DEVICE}")

# Train
model, scaler, marker_cycles, history = train(
    adata,
    cycle_config=DEFAULT_CYCLE_CONFIG,
    n_epochs=N_EPOCHS,
    device_str=DEVICE,
)

# Inference
adata = normalize_adata(
    adata, model, scaler, marker_cycles,
    device_str=DEVICE,
)

print(f"\nDone — adata.layers keys: {list(adata.layers.keys())}")

# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
epochs = range(1, len(history["loss"]) + 1)

axes[0].plot(epochs, history["loss"], "k-", label="Total", linewidth=2)
axes[0].plot(epochs, history["recon"], "b-", alpha=0.7, label="Recon")
axes[0].plot(epochs, history["contrast"], "r-", alpha=0.7, label="Contrast")
axes[0].plot(epochs, history["adv"], "g-", alpha=0.7, label="Adversarial")
axes[0].plot(epochs, history["align"], "m-", alpha=0.7, label="Align")
axes[0].plot(epochs, history["cross_batch"], "c-", alpha=0.7, label="CrossBatch")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss Curves")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history["lr"], "b-")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Learning Rate")
axes[1].set_title("Learning Rate Schedule")
axes[1].set_yscale("log"); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, history["grl_lambda"], "r-")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("GRL Lambda")
axes[2].set_title("Gradient Reversal Ramp")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal losses — total: {history['loss'][-1]:.4f}, "
      f"recon: {history['recon'][-1]:.4f}, "
      f"contrast: {history['contrast'][-1]:.4f}, "
      f"adv: {history['adv'][-1]:.4f}, "
      f"align: {history['align'][-1]:.4f}, "
      f"cross_batch: {history['cross_batch'][-1]:.4f}")

## 3. Verify Output Quality

Check that normalized values are non-negative and marker distributions align across batches.

In [ ]:
X_raw = np.asarray(adata.X) if not sp.issparse(adata.X) else adata.X.toarray()
X_norm = adata.layers["normalized"]

print("=== Output sanity checks ===")
print(f"Non-negative:     {(X_norm >= 0).all()}")
print(f"No NaN:           {not np.isnan(X_norm).any()}")
print(f"No Inf:           {not np.isinf(X_norm).any()}")
print(f"Shape matches:    {X_norm.shape == X_raw.shape}")
print(f"Range: [{X_norm.min():.4f}, {X_norm.max():.4f}]")
print(f"Mean:  {X_norm.mean():.4f}")

In [ ]:
# Per-marker distribution comparison: raw vs normalized across batches
marker_names = list(adata.var_names)
batch_codes = adata.obs["batch"].astype("category").cat.codes.values
batch_names = adata.obs["batch"].astype("category").cat.categories.tolist()
n_batches = len(batch_names)

n_show = min(8, len(marker_names))
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 7), sharey="row")

for i in range(n_show):
    for b in range(n_batches):
        mask = batch_codes == b
        axes[0, i].hist(X_raw[mask, i], bins=40, alpha=0.35, density=True, label=batch_names[b])
        axes[1, i].hist(X_norm[mask, i], bins=40, alpha=0.35, density=True, label=batch_names[b])
    axes[0, i].set_title(f"{marker_names[i]}\n(raw)", fontsize=9)
    axes[1, i].set_title(f"{marker_names[i]}\n(normalized)", fontsize=9)
    if i == 0:
        axes[0, i].set_ylabel("Density")
        axes[1, i].set_ylabel("Density")

axes[0, -1].legend(fontsize=6, loc="upper right")
plt.tight_layout()
plt.show()

## 4. UMAP — Batch Mixing Before/After

Compute UMAP on raw and normalized expression, color by batch. Good normalization should show better batch mixing (overlapping clusters) without losing biological structure.

In [ ]:
import umap

# Subsample for UMAP speed
n_sub = min(50000, adata.n_obs)
rng = np.random.RandomState(42)
sub_idx = rng.choice(adata.n_obs, n_sub, replace=False)

print(f"Computing UMAP on {n_sub} cells (raw)...")
reducer_raw = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
umap_raw = reducer_raw.fit_transform(np.log1p(np.clip(X_raw[sub_idx], 0, None)))

print(f"Computing UMAP on {n_sub} cells (normalized)...")
reducer_norm = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
umap_norm = reducer_norm.fit_transform(np.log1p(np.clip(X_norm[sub_idx], 0, None)))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sc0 = axes[0].scatter(umap_raw[:, 0], umap_raw[:, 1],
                       c=batch_codes[sub_idx], cmap="tab10", s=1, alpha=0.3)
axes[0].set_title("UMAP — Raw (colored by batch)")
axes[0].set_xlabel("UMAP1"); axes[0].set_ylabel("UMAP2")
plt.colorbar(sc0, ax=axes[0], label="batch")

sc1 = axes[1].scatter(umap_norm[:, 0], umap_norm[:, 1],
                       c=batch_codes[sub_idx], cmap="tab10", s=1, alpha=0.3)
axes[1].set_title("UMAP — Normalized (colored by batch)")
axes[1].set_xlabel("UMAP1"); axes[1].set_ylabel("UMAP2")
plt.colorbar(sc1, ax=axes[1], label="batch")

plt.tight_layout()
plt.show()

## 5. KS Statistics — Quantify Batch Effect Reduction

For each marker, compute the pairwise Kolmogorov-Smirnov statistic between all batch pairs. Lower KS = better alignment. Compare raw vs normalized.

In [ ]:
from itertools import combinations

def compute_mean_ks(X, batch_codes, marker_names):
    """Compute mean pairwise KS statistic per marker across all batch pairs."""
    batches = np.unique(batch_codes)
    pairs = list(combinations(batches, 2))
    ks_per_marker = {}

    for m_idx, mname in enumerate(marker_names):
        ks_vals = []
        for b1, b2 in pairs:
            stat, _ = ks_2samp(X[batch_codes == b1, m_idx],
                               X[batch_codes == b2, m_idx])
            ks_vals.append(stat)
        ks_per_marker[mname] = np.mean(ks_vals)
    return ks_per_marker

ks_raw = compute_mean_ks(X_raw, batch_codes, marker_names)
ks_norm = compute_mean_ks(X_norm, batch_codes, marker_names)

# Summary table
print(f"{'Marker':>12s}  {'KS raw':>8s}  {'KS norm':>8s}  {'Change':>8s}")
print("-" * 42)
for m in marker_names:
    change = ks_norm[m] - ks_raw[m]
    arrow = "v" if change < 0 else "^"
    print(f"{m:>12s}  {ks_raw[m]:8.4f}  {ks_norm[m]:8.4f}  {change:+8.4f} {arrow}")

mean_raw = np.mean(list(ks_raw.values()))
mean_norm = np.mean(list(ks_norm.values()))
print(f"\n{'MEAN':>12s}  {mean_raw:8.4f}  {mean_norm:8.4f}  {mean_norm - mean_raw:+8.4f}")
pct_reduction = (1 - mean_norm / mean_raw) * 100
print(f"Overall KS reduction: {pct_reduction:.1f}%")

In [ ]:
# Bar plot comparing KS before/after
x_pos = np.arange(len(marker_names))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
bars1 = ax.bar(x_pos - width/2, [ks_raw[m] for m in marker_names], width, label="Raw", color="salmon")
bars2 = ax.bar(x_pos + width/2, [ks_norm[m] for m in marker_names], width, label="Normalized", color="steelblue")

ax.set_ylabel("Mean pairwise KS statistic")
ax.set_title("Batch effect (KS statistic): Raw vs Normalized")
ax.set_xticks(x_pos)
ax.set_xticklabels(marker_names, rotation=45, ha="right", fontsize=9)
ax.legend()
ax.axhline(y=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## 6. Save Output

In [ ]:
adata.write_h5ad(OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH}")
print(f"  {adata.n_obs} cells x {adata.n_vars} markers")
print(f"  layers: {list(adata.layers.keys())}")

# Download from Colab (uncomment if on Colab)
# from google.colab import files
# files.download(OUTPUT_PATH)